# Convolutional Neural Networks from Scratch: Building Conv2d with NumPy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/convolutional_neural_networks.ipynb)

Implement 2D convolution from scratch in NumPy, verify it matches PyTorch's Conv2d, then train a CNN on MNIST.

**Blog post:** [sesen.ai/blog/convolutional-neural-networks-from-scratch](https://sesen.ai/blog/convolutional-neural-networks-from-scratch)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

## Train a CNN on MNIST

Here is a complete, working CNN that classifies MNIST digits. We will unpack every piece afterwards.

In [ ]:
# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_data = datasets.MNIST('data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('data', train=False, transform=transform)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1000)

In [ ]:
# Define CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=5, padding=2, stride=2)  # 28->14
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1, stride=2)  # 14->7
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=1, stride=2)  # 7->4
        self.fc = nn.Linear(32 * 4 * 4, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        return self.fc(x)

# Train
model = SimpleCNN()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()
    for images, labels in train_loader:
        loss = F.cross_entropy(model(images), labels)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

    # Evaluate
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            correct += (model(images).argmax(1) == labels).sum().item()
    print(f"Epoch {epoch+1}: {correct/len(test_data)*100:.1f}% accuracy")

Five epochs, three convolutional layers, ~98% peak accuracy. Now let's understand what those convolutional layers are actually doing.

## Training Curve

In [ ]:
# Re-train and record metrics for plotting
model = SimpleCNN()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)

train_losses = []
test_accuracies = []

for epoch in range(5):
    model.train()
    epoch_losses = []
    for images, labels in train_loader:
        loss = F.cross_entropy(model(images), labels)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        epoch_losses.append(loss.item())
    train_losses.append(np.mean(epoch_losses))

    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            correct += (model(images).argmax(1) == labels).sum().item()
    acc = correct / len(test_data) * 100
    test_accuracies.append(acc)
    print(f"Epoch {epoch+1}: {acc:.1f}% accuracy, loss={train_losses[-1]:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, 6)

ax1.plot(epochs, test_accuracies, 'o-', color='#2563eb', linewidth=2, markersize=8)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Test Accuracy (%)')
ax1.set_title('Test Accuracy')
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, train_losses, 'o-', color='#dc2626', linewidth=2, markersize=8)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Training Loss')
ax2.set_title('Training Loss')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## The Convolution Operation

A convolution slides a small grid of numbers (the **kernel**) across the input. At each position, it computes element-wise multiplication between the kernel and the overlapping input patch, then sums the results.

In [ ]:
# A 3x3 patch from the input image
patch = np.array([
    [1, 0, 1],
    [0, 1, 0],
    [1, 0, 1]
])

# A 3x3 kernel (edge detector)
kernel = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
])

# Convolution at this position: element-wise multiply, then sum
output_pixel = np.sum(patch * kernel)
print(f"Output pixel value: {output_pixel}")  # 4

## Building Conv2d from Scratch

The full 2D convolution in NumPy — the core function that PyTorch's `nn.Conv2d` runs under the hood.

In [ ]:
def conv2d_numpy(input_img, kernel, stride=1, padding=0):
    """
    2D convolution from scratch.

    Args:
        input_img: shape (H, W) for single channel
        kernel: shape (kH, kW)
        stride: step size for sliding the kernel
        padding: zero-padding added to input borders

    Returns:
        output: shape (out_H, out_W)
    """
    if padding > 0:
        input_img = np.pad(input_img, padding, mode='constant', constant_values=0)

    H, W = input_img.shape
    kH, kW = kernel.shape

    out_H = (H - kH) // stride + 1
    out_W = (W - kW) // stride + 1
    output = np.zeros((out_H, out_W))

    for i in range(out_H):
        for j in range(out_W):
            h_start = i * stride
            w_start = j * stride
            patch = input_img[h_start:h_start + kH, w_start:w_start + kW]
            output[i, j] = np.sum(patch * kernel)

    return output

In [ ]:
# Test on a 6x6 input with a 3x3 edge-detection kernel
input_img = np.array([
    [0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 1, 1, 1, 1, 0],
    [0, 1, 1, 1, 1, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0]
], dtype=np.float32)

edge_kernel = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
], dtype=np.float32)

output = conv2d_numpy(input_img, edge_kernel)
print("Output shape:", output.shape)
print(output)

The output has large positive values (4) along the edges of the diamond where 1s border 0s, and smaller values (1) in the flat interior.

## Edge Detection Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(input_img, cmap='gray')
axes[0].set_title('Input (6x6)')
axes[0].axis('off')

axes[1].imshow(edge_kernel, cmap='RdBu_r', vmin=-8, vmax=8)
axes[1].set_title('Edge Kernel (3x3)')
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{edge_kernel[i,j]:.0f}', ha='center', va='center', fontsize=12)
axes[1].axis('off')

axes[2].imshow(output, cmap='RdBu_r')
axes[2].set_title('Output (4x4)')
for i in range(output.shape[0]):
    for j in range(output.shape[1]):
        axes[2].text(j, i, f'{output[i,j]:.0f}', ha='center', va='center', fontsize=12)
axes[2].axis('off')

plt.tight_layout()
plt.show()

## Verifying Against PyTorch

The real test: does our NumPy implementation produce the same output as PyTorch?

In [ ]:
# Same input and kernel
input_torch = torch.tensor(input_img).unsqueeze(0).unsqueeze(0)  # (1, 1, 6, 6)
kernel_torch = torch.tensor(edge_kernel).unsqueeze(0).unsqueeze(0)  # (1, 1, 3, 3)

output_torch = F.conv2d(input_torch, kernel_torch)
output_numpy = conv2d_numpy(input_img, edge_kernel)

print("PyTorch output:\n", output_torch.squeeze().numpy())
print("\nNumPy output:\n", output_numpy)
print("\nMatch:", np.allclose(output_numpy, output_torch.squeeze().numpy()))

## The Output Size Formula

$$H_{\text{out}} = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1$$

In [ ]:
H_out = (28 + 2*2 - 5) // 2 + 1  # = 14
print(f"Output size: {H_out}x{H_out}")  # 14x14

## Stride and Padding

In [ ]:
# stride=1, no padding: output shrinks
out1 = conv2d_numpy(input_img, edge_kernel, stride=1, padding=0)
print(f"No padding:  {input_img.shape} -> {out1.shape}")  # (6,6) -> (4,4)

# stride=1, padding=1: output same size
out2 = conv2d_numpy(input_img, edge_kernel, stride=1, padding=1)
print(f"Padding=1:   {input_img.shape} -> {out2.shape}")  # (6,6) -> (6,6)

# stride=2, padding=1: output halved
out3 = conv2d_numpy(input_img, edge_kernel, stride=2, padding=1)
print(f"Stride=2:    {input_img.shape} -> {out3.shape}")  # (6,6) -> (3,3)

## Multi-Channel Convolution

Real images have multiple channels. Multi-channel convolution applies a separate kernel to each input channel and sums the results.

In [ ]:
def conv2d_multichannel(input_img, kernels, bias=None, stride=1, padding=0):
    """
    Multi-channel convolution: (C_in, H, W) input, (C_out, C_in, kH, kW) kernels.

    Each output channel is the sum of convolutions across all input channels,
    plus an optional bias term.
    """
    C_out, C_in, kH, kW = kernels.shape

    if padding > 0:
        input_img = np.pad(input_img, ((0, 0), (padding, padding), (padding, padding)),
                           mode='constant', constant_values=0)

    _, H, W = input_img.shape
    out_H = (H - kH) // stride + 1
    out_W = (W - kW) // stride + 1
    output = np.zeros((C_out, out_H, out_W))

    for co in range(C_out):
        for ci in range(C_in):
            for i in range(out_H):
                for j in range(out_W):
                    h_start = i * stride
                    w_start = j * stride
                    patch = input_img[ci, h_start:h_start+kH, w_start:w_start+kW]
                    output[co, i, j] += np.sum(patch * kernels[co, ci])
        if bias is not None:
            output[co] += bias[co]

    return output

In [ ]:
# Verify multi-channel convolution matches PyTorch
np.random.seed(42)
input_mc = np.random.randn(3, 8, 8).astype(np.float32)

# Random kernels: 4 output channels, 3 input channels, 3x3
kernels = np.random.randn(4, 3, 3, 3).astype(np.float32)
bias = np.random.randn(4).astype(np.float32)

# NumPy version
out_np = conv2d_multichannel(input_mc, kernels, bias, stride=1, padding=1)

# PyTorch version
input_t = torch.tensor(input_mc).unsqueeze(0)
kernel_t = torch.tensor(kernels)
bias_t = torch.tensor(bias)
out_pt = F.conv2d(input_t, kernel_t, bias_t, stride=1, padding=1).squeeze(0).numpy()

print(f"Shape: {out_np.shape}")
print(f"Match: {np.allclose(out_np, out_pt, atol=1e-5)}")

## What the Kernels Learn

Different kernels detect different features. Let's visualise classic hand-crafted kernels applied to an MNIST digit.

In [ ]:
# Horizontal edge detector
horizontal = np.array([[-1, -1, -1],
                        [ 0,  0,  0],
                        [ 1,  1,  1]], dtype=np.float32)

# Vertical edge detector
vertical = np.array([[-1, 0, 1],
                      [-1, 0, 1],
                      [-1, 0, 1]], dtype=np.float32)

# Sharpen filter
sharpen = np.array([[ 0, -1,  0],
                     [-1,  5, -1],
                     [ 0, -1,  0]], dtype=np.float32)

# Gaussian blur (3x3 approximation)
blur = np.array([[1, 2, 1],
                  [2, 4, 2],
                  [1, 2, 1]], dtype=np.float32) / 16

# Apply to an MNIST digit
sample = test_data[0][0].squeeze().numpy()

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
titles = ['Original', 'Horizontal edges', 'Vertical edges', 'Sharpen', 'Blur']
results = [sample,
           conv2d_numpy(sample, horizontal, padding=1),
           conv2d_numpy(sample, vertical, padding=1),
           conv2d_numpy(sample, sharpen, padding=1),
           conv2d_numpy(sample, blur, padding=1)]

for ax, img, title in zip(axes, results, titles):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Max Pooling

In [ ]:
def max_pool2d(input_img, pool_size=2, stride=2):
    """Max pooling: take the maximum value in each pool_size x pool_size patch."""
    H, W = input_img.shape
    out_H = (H - pool_size) // stride + 1
    out_W = (W - pool_size) // stride + 1
    output = np.zeros((out_H, out_W))

    for i in range(out_H):
        for j in range(out_W):
            h_start, w_start = i * stride, j * stride
            output[i, j] = np.max(input_img[h_start:h_start+pool_size,
                                             w_start:w_start+pool_size])
    return output

# Demo: apply max pooling to an MNIST digit
pooled = max_pool2d(sample)
print(f"Original: {sample.shape} -> Pooled: {pooled.shape}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(sample, cmap='gray')
ax1.set_title(f'Original {sample.shape}')
ax1.axis('off')
ax2.imshow(pooled, cmap='gray')
ax2.set_title(f'Max Pooled {pooled.shape}')
ax2.axis('off')
plt.tight_layout()
plt.show()

## CNN with Batch Normalisation

Adding batch norm after each conv layer stabilises training and allows higher learning rates.

In [ ]:
class CNNWithBatchNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(1, 8, 5, padding=2, stride=2), nn.BatchNorm2d(8), nn.ReLU(),
            nn.Conv2d(8, 16, 3, padding=1, stride=2), nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1, stride=2), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1, stride=2), nn.BatchNorm2d(32), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32, 10)
        )

    def forward(self, x):
        return self.layers(x)

## Visualising Feature Maps

To see what a trained CNN "sees", we extract the activations after each conv layer.

In [ ]:
def get_feature_maps(model, image):
    """Extract feature maps from each conv layer."""
    activations = []
    x = image.unsqueeze(0)

    for layer in [model.conv1, model.conv2, model.conv3]:
        x = F.relu(layer(x))
        activations.append(x.squeeze(0).detach().numpy())

    return activations

# Get feature maps for a single digit
sample_image = test_data[0][0]  # First test image
feature_maps = get_feature_maps(model, sample_image)

for i, fmaps in enumerate(feature_maps):
    print(f"Layer {i+1}: {fmaps.shape[0]} channels, {fmaps.shape[1]}x{fmaps.shape[2]} spatial")

In [ ]:
# Visualise feature maps from each layer
fig, axes = plt.subplots(3, 8, figsize=(16, 6))

for layer_idx, fmaps in enumerate(feature_maps):
    n_channels = min(8, fmaps.shape[0])
    for ch in range(8):
        if ch < n_channels:
            axes[layer_idx, ch].imshow(fmaps[ch], cmap='viridis')
        axes[layer_idx, ch].axis('off')
        if ch == 0:
            axes[layer_idx, ch].set_ylabel(f'Layer {layer_idx+1}', fontsize=12)

plt.suptitle('Feature Maps from Each Convolutional Layer', fontsize=14)
plt.tight_layout()
plt.show()

## Exercises

1. **Custom kernel** — Design a 3x3 kernel that detects diagonal edges (top-left to bottom-right). Apply it to an MNIST digit and verify visually.

2. **Stride experiment** — Modify `SimpleCNN` to use stride 1 everywhere and add `nn.MaxPool2d(2)` after each ReLU instead. Compare accuracy and training speed.

3. **Depth vs width** — Train two networks: one with double the filters (16, 32, 64) and one with an extra conv layer (8, 16, 32, 32). Which gets higher accuracy? Which trains faster?

4. **Batch norm effect** — Train `CNNWithBatchNorm` at learning rate 0.01 (10x higher than our default). Does it converge? Try the same learning rate with `SimpleCNN`.

5. **Visualise learned kernels** — After training, plot `model.conv1.weight.data` as 8 small images. Do any resemble the hand-crafted edge detectors?

## References

- LeCun, Y., Bottou, L., Bengio, Y., & Haffner, P. (1998). [Gradient-Based Learning Applied to Document Recognition.](http://yann.lecun.com/exdb/publis/pdf/lecun-01a.pdf) Proceedings of the IEEE, 86(11), 2278-2324.
- Fukushima, K. (1980). Neocognitron: A Self-organizing Neural Network Model. Biological Cybernetics, 36, 193-202.
- fast.ai course: [Practical Deep Learning for Coders](https://course.fast.ai/), Lessons 7-8.